# Gradient Boosting — functional gradient descent with trees

> Tutorial pair for [`gradient_boosting.py`](gradient_boosting.py).

## 1. Intuition
Boosting builds the model **one tree at a time**. Start with a constant guess.
Look at where you're wrong — the *residuals* — and fit a small tree to those
errors. Add a shrunken copy of that tree to your running prediction and repeat.
Each tree nudges the prediction down the loss surface, so the ensemble keeps
*reducing bias* (the opposite emphasis from a random forest, which reduces
variance). This "fit the residual" recipe is exactly gradient descent — but in
the space of *functions*.

## 2. Concept (the slide)
- **Additive model:** $F_M(x)=F_0(x)+\nu\sum_{m=1}^{M} h_m(x)$, each $h_m$ a
  shallow regression tree.
- **Stagewise:** freeze previous trees; the new tree fits the **negative
  gradient** of the loss at the current predictions (the *pseudo-residuals*).
- **Losses:** squared error → residuals $y-F$ (regression); logistic/deviance →
  $y-\sigma(F)$ (classification).
- **Shrinkage** $\nu$ (learning rate): take small steps → better generalization
  (needs more trees).
- **Stochastic GB:** subsample rows per round for speed + extra regularization.
- **Newton / XGBoost:** use the **second** derivative (Hessian) and an L2 penalty
  on leaf weights for a regularized, faster-converging step.

## 3. Math derivation

**Functional gradient descent.** We want $F$ minimizing
$\mathcal L(F)=\sum_{i=1}^n \ell(y_i, F(x_i))$. Treat the vector of predictions
$\big(F(x_1),\dots,F(x_n)\big)$ as the parameters. Gradient descent would update
$F(x_i)\leftarrow F(x_i)-\nu\,g_i$ with the **functional gradient**

$$g_i=\frac{\partial \ell(y_i,F(x_i))}{\partial F(x_i)} .$$

But that only updates the values at training points. To **generalize**, we fit a
regression tree $h_m$ to the negative gradient (the *pseudo-residuals*
$r_i=-g_i$) and step in that direction:

$$F_m = F_{m-1} + \nu\, h_m,\qquad h_m \approx \arg\min_h \sum_i \big(r_i-h(x_i)\big)^2 .$$

**Squared error** $\ell=\tfrac12(y-F)^2$: $-g_i = y_i-F_{m-1}(x_i)$ — the
ordinary **residual**. So least-squares boosting literally fits each tree to the
residuals, and $F_0=\bar y$ (the minimizer of constant loss).

**Logistic loss** for $y\in\{0,1\}$ with $\ell=-[y\log p+(1-y)\log(1-p)]$,
$p=\sigma(F)$: one finds $\partial\ell/\partial F = \sigma(F)-y$, hence
$-g_i = y_i-\sigma(F_{m-1}(x_i))$, and $F_0=\log\frac{\bar y}{1-\bar y}$ (the
log-odds). Predictions come from $\sigma(F_M)$.

**Shrinkage.** Replacing $F_m=F_{m-1}+h_m$ with $F_m=F_{m-1}+\nu h_m$, $\nu\in(0,1]$,
is a learning rate. Small $\nu$ (e.g. $0.1$) regularizes — it prevents any single
tree from dominating and empirically lowers test error, at the cost of needing
more trees ($M\propto 1/\nu$).

**Newton boosting (XGBoost).** Second-order Taylor expansion of the loss around
$F_{m-1}$ with per-sample gradient $g_i$ and Hessian $h_i=\partial^2\ell/\partial F^2$:

$$\mathcal L(F_{m-1}+f)\approx \text{const}+\sum_i\Big(g_i f(x_i)+\tfrac12 h_i f(x_i)^2\Big)
  +\tfrac12\lambda\sum_j w_j^2 .$$

For a tree with leaves $j$ (region $I_j$), $f$ is constant $w_j$ on each leaf. Let
$G_j=\sum_{i\in I_j} g_i,\ H_j=\sum_{i\in I_j} h_i$. Minimizing the quadratic in
$w_j$ gives the **optimal leaf weight and value**

$$\boxed{\,w_j^\star=-\frac{G_j}{H_j+\lambda}\,},\qquad
  \mathcal L^\star=-\tfrac12\sum_j\frac{G_j^2}{H_j+\lambda}.$$

The **split gain** for partitioning a node into $L,R$ is therefore

$$\text{Gain}=\tfrac12\!\left[\frac{G_L^2}{H_L+\lambda}+\frac{G_R^2}{H_R+\lambda}
   -\frac{(G_L{+}G_R)^2}{H_L{+}H_R+\lambda}\right]-\gamma,$$

with $\gamma$ a minimum-gain (complexity) penalty. For squared loss $h_i\equiv1$
and Newton reduces to ordinary residual fitting; for logistic loss
$h_i=p_i(1-p_i)$ gives the curvature-aware step.

## 4. NumPy implementation (GBM regression + classification + Newton/XGBoost)

In [ ]:
# ===== actual implementation from gradient_boosting.py =====
from __future__ import annotations

import numpy as np

SEED = 0

class _RegNode:
    __slots__ = ("feature", "threshold", "left", "right", "value")

    def __init__(self):
        self.feature = self.threshold = self.left = self.right = self.value = None

class _RegTree:
    """Shallow CART regressor used as the weak learner inside boosting."""

    def __init__(self, max_depth=3, min_samples_split=2, lam=0.0, gamma=0.0,
                 mode="variance"):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.lam = lam            # L2 leaf regularization (Newton mode)
        self.gamma = gamma        # min split gain (Newton mode)
        self.mode = mode

    # ----- variance (squared-error) splitting -----
    # Vectorized via prefix sums: for a feature sorted ascending, every split is
    # a prefix/suffix. SSE = sum(y^2) - (sum y)^2 / count, so total child SSE for
    # all thresholds at once costs O(n) after the O(n log n) sort.
    def _best_split_var(self, X, y):
        n, d = X.shape
        best = (np.inf, None, None)   # minimize total child SSE
        for f in range(d):
            order = np.argsort(X[:, f], kind="mergesort")
            xs, ys = X[order, f], y[order]
            cs = np.cumsum(ys)          # prefix sum of y
            cs2 = np.cumsum(ys ** 2)    # prefix sum of y^2
            tot, tot2 = cs[-1], cs2[-1]
            cnt = np.arange(1, n)       # left sizes 1..n-1
            sl, sl2 = cs[:-1], cs2[:-1]
            sr, sr2 = tot - sl, tot2 - sl2
            sse_l = sl2 - sl ** 2 / cnt
            sse_r = sr2 - sr ** 2 / (n - cnt)
            total = sse_l + sse_r
            # only valid where the feature value actually changes (real split)
            valid = xs[:-1] != xs[1:]
            total = np.where(valid, total, np.inf)
            j = int(np.argmin(total))
            if total[j] < best[0]:
                best = (total[j], f, (xs[j] + xs[j + 1]) / 2)
        return best

    # ----- Newton (gradient/Hessian) splitting -----
    @staticmethod
    def _leaf_weight(g, h, lam):
        # closed-form minimizer of  G*w + 1/2 (H+lam) w^2  ->  w = -G/(H+lam)
        return -g.sum() / (h.sum() + lam)

    @staticmethod
    def _struct_gain(g, h, lam):
        # XGBoost structure score for a node:  1/2 * G^2 / (H + lam)
        return 0.5 * g.sum() ** 2 / (h.sum() + lam)

    def _best_split_newton(self, X, g, h):
        # Vectorized XGBoost gain over every threshold via prefix sums of g, h.
        n, d = X.shape
        lam = self.lam
        parent = self._struct_gain(g, h, lam)
        best = (-np.inf, None, None)  # maximize gain
        for f in range(d):
            order = np.argsort(X[:, f], kind="mergesort")
            xs, gs, hs = X[order, f], g[order], h[order]
            Gl, Hl = np.cumsum(gs)[:-1], np.cumsum(hs)[:-1]
            Gr, Hr = g.sum() - Gl, h.sum() - Hl
            gain = 0.5 * (Gl ** 2 / (Hl + lam) + Gr ** 2 / (Hr + lam)) - parent - self.gamma
            valid = xs[:-1] != xs[1:]
            gain = np.where(valid, gain, -np.inf)
            j = int(np.argmax(gain))
            if gain[j] > best[0]:
                best = (gain[j], f, (xs[j] + xs[j + 1]) / 2)
        return best

    def _build(self, X, target, depth, g=None, h=None):
        node = _RegNode()
        n = len(X)
        if self.mode == "newton":
            if n < self.min_samples_split or depth >= self.max_depth:
                node.value = self._leaf_weight(g, h, self.lam)
                return node
            gain, f, t = self._best_split_newton(X, g, h)
            if f is None or gain <= 0:        # gain<=0 means split not worth it
                node.value = self._leaf_weight(g, h, self.lam)
                return node
            m = X[:, f] <= t
            node.feature, node.threshold = f, t
            node.left = self._build(X[m], None, depth + 1, g[m], h[m])
            node.right = self._build(X[~m], None, depth + 1, g[~m], h[~m])
            return node
        # variance mode (fits `target` = pseudo-residuals)
        if n < self.min_samples_split or depth >= self.max_depth or np.var(target) == 0:
            node.value = float(np.mean(target))
            return node
        _, f, t = self._best_split_var(X, target)
        if f is None:
            node.value = float(np.mean(target))
            return node
        m = X[:, f] <= t
        node.feature, node.threshold = f, t
        node.left = self._build(X[m], target[m], depth + 1)
        node.right = self._build(X[~m], target[~m], depth + 1)
        return node

    def fit(self, X, target=None, g=None, h=None):
        X = np.asarray(X, float)
        self.root = self._build(X, target, 0, g, h)
        return self

    def _predict_one(self, x, node):
        if node.value is not None:
            return node.value
        branch = node.left if x[node.feature] <= node.threshold else node.right
        return self._predict_one(x, branch)

    def predict(self, X):
        return np.array([self._predict_one(x, self.root)
                         for x in np.asarray(X, float)])

def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def demo():
    np.random.seed(SEED)
    from sklearn.datasets import make_friedman1, make_classification

    # ---------- regression (squared loss) ----------
    Xr, yr = make_friedman1(n_samples=400, noise=1.0, random_state=SEED)
    Xtr, ytr, Xte, yte = Xr[:320], yr[:320], Xr[320:], yr[320:]
    gbr = GradientBoostingNumPy(loss="squared", n_estimators=200,
                                learning_rate=0.1, max_depth=3).fit(Xtr, ytr)
    print(f"[reg] GBM MSE={np.mean((gbr.predict(Xte) - yte) ** 2):.3f}  "
          f"(final train loss {gbr.train_loss_[-1]:.3f})")
    skr = sklearn_reference(Xtr, ytr, task="regression",
                            n_estimators=200, learning_rate=0.1, max_depth=3)
    print(f"[reg] sklearn GBM MSE={np.mean((skr.predict(Xte) - yte) ** 2):.3f}")

    # ---------- binary classification (logistic loss) ----------
    Xc, yc = make_classification(n_samples=500, n_features=12, n_informative=6,
                                 n_redundant=2, random_state=SEED)
    Xctr, yctr, Xcte, ycte = Xc[:380], yc[:380], Xc[380:], yc[380:]
    gbc = GradientBoostingNumPy(loss="logistic", method="gradient",
                                n_estimators=150, learning_rate=0.1,
                                max_depth=3).fit(Xctr, yctr)
    print(f"[clf] GBM (gradient) acc={np.mean(gbc.predict(Xcte) == ycte):.3f}")

    # ---------- second-order / regularized (XGBoost-style) ----------
    gbn = GradientBoostingNumPy(loss="logistic", method="newton",
                                n_estimators=150, learning_rate=0.1,
                                max_depth=3, lam=1.0, gamma=0.0).fit(Xctr, yctr)
    print(f"[clf] GBM (Newton/XGB) acc={np.mean(gbn.predict(Xcte) == ycte):.3f}")

    # stochastic gradient boosting
    gbs = GradientBoostingNumPy(loss="logistic", n_estimators=150,
                                learning_rate=0.1, subsample=0.6).fit(Xctr, yctr)
    print(f"[clf] GBM (subsample=0.6) acc={np.mean(gbs.predict(Xcte) == ycte):.3f}")

    skc = sklearn_reference(Xctr, yctr, task="classification",
                            n_estimators=150, learning_rate=0.1, max_depth=3)
    print(f"[clf] sklearn GBM acc={np.mean(skc.predict(Xcte) == ycte):.3f}")


class GradientBoostingNumPy:
    """Stagewise additive boosting of shallow regression trees.

    loss="squared"  -> regression (pseudo-residuals = y - F)
    loss="logistic" -> binary classification (deviance / log-loss)

    method="gradient" : first-order (Friedman) — fit tree to negative gradient.
    method="newton"   : second-order (XGBoost) — fit tree using g & h with a
                        regularized closed-form leaf weight w* = -G/(H+lambda).
    """

    def __init__(self, loss="squared", method="gradient", n_estimators=100,
                 learning_rate=0.1, max_depth=3, subsample=1.0,
                 lam=1.0, gamma=0.0, seed=SEED):
        self.loss = loss
        self.method = method
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.subsample = subsample        # stochastic GB: fraction of rows / round
        self.lam = lam
        self.gamma = gamma
        self.seed = seed
        self.trees_ = []
        self.train_loss_ = []

    # ---- loss-specific quantities ----
    def _init_raw(self, y):
        # F_0 = argmin_c sum loss(y, c)
        if self.loss == "squared":
            return float(np.mean(y))                  # mean minimizes MSE
        p = np.clip(np.mean(y), 1e-6, 1 - 1e-6)
        return float(np.log(p / (1 - p)))             # log-odds minimizes log-loss

    def _neg_gradient(self, y, F):
        # -dL/dF  (the pseudo-residual)
        if self.loss == "squared":
            return y - F                              # residual
        return y - _sigmoid(F)                        # logistic: y - p

    def _hessian(self, y, F):
        # d^2 L / dF^2
        if self.loss == "squared":
            return np.ones_like(F)
        p = _sigmoid(F)
        return np.clip(p * (1 - p), 1e-6, None)

    def _loss_value(self, y, F):
        if self.loss == "squared":
            return float(np.mean((y - F) ** 2))
        p = _sigmoid(F)
        return float(-np.mean(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12)))

    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y, float)
        n = len(X)
        rng = np.random.default_rng(self.seed)
        self.F0_ = self._init_raw(y)
        F = np.full(n, self.F0_)
        self.trees_, self.train_loss_ = [], []

        for _ in range(self.n_estimators):
            # stochastic subsampling of rows (Friedman 2002)
            if self.subsample < 1.0:
                idx = rng.choice(n, max(1, int(self.subsample * n)), replace=False)
            else:
                idx = np.arange(n)

            if self.method == "newton":
                # second order: gradient g = dL/dF, hessian h = d^2L/dF^2
                g = -self._neg_gradient(y[idx], F[idx])   # note: g = dL/dF = -(neg grad)
                h = self._hessian(y[idx], F[idx])
                tree = _RegTree(max_depth=self.max_depth, lam=self.lam,
                                gamma=self.gamma, mode="newton").fit(X[idx], g=g, h=h)
            else:
                # first order: fit tree to the negative gradient (pseudo-residual)
                resid = self._neg_gradient(y[idx], F[idx])
                tree = _RegTree(max_depth=self.max_depth,
                                mode="variance").fit(X[idx], target=resid)

            # shrinkage: F <- F + nu * tree(X)  (learning rate regularizes)
            F += self.learning_rate * tree.predict(X)
            self.trees_.append(tree)
            self.train_loss_.append(self._loss_value(y, F))
        return self

    def decision_function(self, X):
        """Raw additive score F(x) = F0 + nu * sum_m tree_m(x)."""
        F = np.full(len(np.asarray(X, float)), self.F0_)
        for tree in self.trees_:
            F += self.learning_rate * tree.predict(X)
        return F

    def predict_proba(self, X):
        assert self.loss == "logistic"
        p = _sigmoid(self.decision_function(X))
        return np.column_stack([1 - p, p])

    def predict(self, X):
        F = self.decision_function(X)
        if self.loss == "logistic":
            return (_sigmoid(F) >= 0.5).astype(int)
        return F

## 5. Reference / cross-check — why not PyTorch?

The "gradient" in gradient boosting is the **functional** gradient of the loss
w.r.t. the model's predictions — computed in closed form, not via autograd. The
weak learners are *discrete, greedily-grown* regression trees, which are
non-differentiable, so an idiomatic PyTorch model is not the natural tool. We
cross-check against scikit-learn's `GradientBoosting*`.

In [ ]:
# ===== actual implementation from gradient_boosting.py =====
def sklearn_reference(X, y, task="classification", **kw):
    from sklearn.ensemble import (GradientBoostingClassifier,
                                   GradientBoostingRegressor)
    Model = (GradientBoostingClassifier if task == "classification"
             else GradientBoostingRegressor)
    return Model(random_state=SEED, **kw).fit(X, y)

## 6. Train — regression, logistic classification, Newton, and stochastic GB

In [ ]:
demo()

## 7. Visualization — loss decreasing per stage; shrinkage effect

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_friedman1
import gradient_boosting as M

Xr, yr = make_friedman1(n_samples=400, noise=1.0, random_state=0)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for lr in (0.03, 0.1, 0.3, 1.0):
    gb = M.GradientBoostingNumPy(loss="squared", n_estimators=150,
                                 learning_rate=lr, max_depth=3).fit(Xr, yr)
    ax[0].plot(gb.train_loss_, label=f"lr={lr}")
ax[0].set_xlabel("boosting round"); ax[0].set_ylabel("train MSE")
ax[0].set_title("Shrinkage: smaller lr = slower, smoother descent"); ax[0].legend()

# first-order vs Newton on a classification problem
from sklearn.datasets import make_classification
Xc, yc = make_classification(n_samples=400, n_features=10, n_informative=6,
                             random_state=0)
g1 = M.GradientBoostingNumPy(loss="logistic", method="gradient",
                             n_estimators=120, learning_rate=0.1).fit(Xc, yc)
g2 = M.GradientBoostingNumPy(loss="logistic", method="newton",
                             n_estimators=120, learning_rate=0.1, lam=1.0).fit(Xc, yc)
ax[1].plot(g1.train_loss_, label="first-order (gradient)")
ax[1].plot(g2.train_loss_, label="second-order (Newton/XGB)")
ax[1].set_xlabel("boosting round"); ax[1].set_ylabel("train log-loss")
ax[1].set_title("Newton step converges faster"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- Boosting **reduces bias** by sequentially correcting errors — complementary to
  bagging/forests (which reduce variance). It can overfit if you boost too long.
- The trio **(learning rate, n_estimators, max_depth)** is the core trade-off:
  small $\nu$ + many shallow trees + early stopping generalizes best.
- Sequential ⇒ **not embarrassingly parallel** like a forest; sensitive to
  noisy labels (it chases them).
- **Newton/XGBoost** uses curvature and an L2 leaf penalty $\lambda$ for a more
  regularized, faster-converging step; add $\gamma$ to prune low-gain splits.
- Always tune on a validation curve; the training loss alone keeps dropping.